## EXPERIMENTACIÓN — EXP_BD01: Silver vs Gold (arquitectura Medallion)
**Objetivo:** evaluar si cambiar la arquitectura de consulta reduce el tiempo de respuesta: calcular los indicadores Q1–Q4 desde la tabla limpia (**Silver**) cada vez, o leerlos precalculados desde tablas **Gold**; y estimar cuántas consultas se necesitan para amortizar el costo de construir Gold.

**Volúmenes:** ×1 = dataset original (180,519 filas) · ×10 = ese mismo dataset replicado 10 veces (1,805,190 filas, datos sintéticos).
**Criterios del experimento:** resultados de Gold idénticos a Silver · medición estable (CV ≤ 15 %) · mejora en tiempo respecto a Silver.

Requisito: haber ejecutado antes el notebook de la solución, que crea la tabla `tb_supply_chain`.

In [ ]:
# PREPARACIÓN
import time, platform
import pandas as pd
from pyspark.sql import functions as F

dbutils.widgets.text("codEstudiante", "")
codEstudiante = dbutils.widgets.get("codEstudiante")
SUF = f"_{codEstudiante}" if codEstudiante else ""

N_REP     = 5
UMBRAL_CV = 15                       # % máximo de variación entre repeticiones
FACTORES  = [1, 10]
SILVER, GOLD = "Silver (calcula cada vez)", "Gold (precalculada)"
Q = ["Q1", "Q2", "Q3", "Q4"]

TABLA_BASE = f"tb_supply_chain{SUF}"
SILVER_T = {k: (TABLA_BASE if k == 1 else f"exp_bd01_silver_x{k}{SUF}") for k in FACTORES}

# Tablas Silver: ×1 es la original; ×k replica el dataset k veces con order_id distinto
for k in FACTORES:
    if k > 1:
        (spark.table(TABLA_BASE)
              .withColumn("rep", F.explode(F.sequence(F.lit(1), F.lit(k))))
              .withColumn("order_id", F.col("order_id") + (F.col("rep") - 1) * 1_000_000)
              .drop("rep").write.mode("overwrite").saveAsTable(SILVER_T[k]))

N_FILAS = {k: spark.table(SILVER_T[k]).count() for k in FACTORES}
for k in FACTORES:
    dup = spark.table(SILVER_T[k]).groupBy("order_id", "order_item_id").count().filter("count > 1").count()
    print(f"×{k}: {N_FILAS[k]:,} filas | claves duplicadas = {dup}")
print({"spark_version": spark.version, "python": platform.python_version(), "fecha": time.strftime("%Y-%m-%d %H:%M:%S")})

def repetir(k, q, alternativa, fn):
    fn()                                              # calentamiento (se descarta)
    regs = []
    for r in range(1, N_REP + 1):
        t0 = time.perf_counter(); fn(); seg = time.perf_counter() - t0
        regs.append({"factor": k, "consulta": q, "alternativa": alternativa, "repeticion": r,
                     "filas": N_FILAS[k], "segundos": round(seg, 4)})
    print(f"×{k} {q} | {alternativa}: " + ", ".join(f"{x['segundos']:.2f}s" for x in regs))
    return regs

In [ ]:
# Indicadores Q1-Q4 (mismas definiciones que la solución BI) y construcción de las tablas Gold
def consultas(t):
    return {
      "Q1": t.agg(F.count("*").alias("total"), F.sum("late_delivery_risk").alias("tardias"),
                  F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"),
                  F.round(F.avg("days_for_shipping_real"), 2).alias("dias_reales"),
                  F.round(F.avg("days_for_shipment_scheduled"), 2).alias("dias_programados"),
                  F.round(F.sum("sales"), 2).alias("ventas")),
      "Q2": t.groupBy("shipping_mode").agg(F.count("*").alias("items"), F.sum("late_delivery_risk").alias("tardias"),
                  F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias")),
      "Q3": t.groupBy("order_region", "category_name").agg(F.sum("late_delivery_risk").alias("tardias")),
      "Q4": t.groupBy("order_year", "order_month").agg(F.sum("late_delivery_risk").alias("tardias"),
                  F.round(F.sum("order_profit_per_order"), 2).alias("ganancia"))}

nombre_gold = lambda k, q: f"exp_bd01_gold_{q.lower()}_x{k}{SUF}"

construccion = {}
for k in FACTORES:
    for q, dfq in consultas(spark.table(SILVER_T[k])).items():
        t0 = time.perf_counter()
        dfq.write.mode("overwrite").saveAsTable(nombre_gold(k, q))
        construccion[(k, q)] = round(time.perf_counter() - t0, 2)
print("Construcción de Gold (s):", construccion)

# Validación: Gold debe contener exactamente lo mismo que calcular desde Silver
filas = lambda x: sorted(tuple(r) for r in x.collect())
ok = all(filas(consultas(spark.table(SILVER_T[k]))[q]) == filas(spark.table(nombre_gold(k, q))) for k in FACTORES for q in Q)
print("Gold idéntico a Silver:", ok)

In [ ]:
# Medición (1 calentamiento + 5 repeticiones por alternativa)
reg = []
for k in FACTORES:
    s = spark.table(SILVER_T[k])
    for q in Q:
        reg += repetir(k, q, SILVER, lambda: consultas(s)[q].collect())
        reg += repetir(k, q, GOLD,   lambda: spark.table(nombre_gold(k, q)).collect())

reg_pd = pd.DataFrame(reg)
res = (reg_pd.groupby(["factor", "consulta", "alternativa"])["segundos"]
             .agg(mediana="median", media="mean", desv="std", minimo="min", maximo="max").reset_index())
res["cv_pct"] = (res["desv"] / res["media"] * 100).round(2)
res = res.round(4)
display(res)

In [ ]:
# Evaluación: estabilidad y mejora de Gold frente a Silver (mismo volumen y consulta)
med = res.set_index(["factor", "consulta", "alternativa"])["mediana"]
res["estable"]    = res["cv_pct"] <= UMBRAL_CV
res["mejora_pct"] = res.apply(lambda r: round((1 - r["mediana"] / med[(r["factor"], r["consulta"], SILVER)]) * 100, 1), axis=1)
display(res[["factor", "consulta", "alternativa", "mediana", "cv_pct", "estable", "mejora_pct"]])

# ¿Cuántas consultas hacen falta para amortizar la construcción de Gold?
filas_am = []
for k in FACTORES:
    for q in Q:
        ahorro = med[(k, q, SILVER)] - med[(k, q, GOLD)]
        filas_am.append({"factor": k, "consulta": q, "construccion_gold_s": construccion[(k, q)],
                         "ahorro_por_consulta_s": round(ahorro, 4),
                         "consultas_para_amortizar": round(construccion[(k, q)] / ahorro, 1) if ahorro > 0 else float("nan")})
amort = pd.DataFrame(filas_am)
display(amort)

In [ ]:
# Gráfico: tiempo por consulta, Silver vs Gold, para cada volumen
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(FACTORES), figsize=(11, 4))
for ax, k in zip(axes, FACTORES):
    g = res[res["factor"] == k].pivot(index="consulta", columns="alternativa", values="mediana")[[SILVER, GOLD]]
    g.plot(kind="bar", ax=ax)
    ax.set_title(f"×{k} ({N_FILAS[k]:,} filas)"); ax.set_xlabel("Consulta"); ax.set_ylabel("Segundos (mediana)")
    ax.grid(axis="y", alpha=.3); ax.tick_params(axis="x", rotation=0)
plt.suptitle("EXP_BD01: Silver vs Gold"); plt.tight_layout(); plt.show()

In [ ]:
# Guardado de resultados (tablas Delta + CSV en el Volume)
def guardar(pdf, nombre):
    spark.createDataFrame(pdf).write.mode("overwrite").saveAsTable(nombre + SUF)
    ruta = f"/Volumes/workspace/default/datos/{nombre}{SUF}.csv"
    pdf.to_csv(ruta, index=False)
    print("Guardado:", nombre + SUF, "y", ruta)

guardar(reg_pd, "exp_bd01_mediciones")
guardar(res,    "exp_bd01_resumen")
guardar(amort,  "exp_bd01_amortizacion")